## Setup

In [1]:
import torch
import numpy as np
import pandas as pd
# import plotly.express as px
import plotly.graph_objects as go

from notebooks.intervention.intervene import models, tgt2s, src_src, tgt_tgt

# Axis labels
src_labels = [f"{s['src_lang_base']}-{s['src_lang_source']}" for s in src_src]
tgt_labels = [f"{t['tgt_lang_base']}-{t['tgt_lang_source']}" for t in tgt_tgt]

postfix = "_complete"

In [ ]:
def get_earliest_layer(dataframe, column, val_base, val_source):

    # TODO: standardize the token_type
    dataframe['part_of_speech'] = dataframe['token_type'].apply(lambda x: x.split('-')[0])
    example = dataframe['token_type'][0]
    if example.split('-')[1] in ("base", "source"):
        dataframe['lexical_component'] = dataframe['token_type'].apply(lambda x: x.split('-')[1])
        dataframe['language'] = dataframe['token_type'].apply(lambda x: x.split('-')[2])
    elif example.split('-')[2] in ("base", "source"):
        dataframe['language'] = dataframe['token_type'].apply(lambda x: x.split('-')[1])
        dataframe['lexical_component'] = dataframe['token_type'].apply(lambda x: x.split('-')[2])
    df_heatmap = dataframe.pivot_table(
        index="layer",
        columns=column,
        values="prob",
        aggfunc="sum",
        fill_value=0
    )
    # print(val_base)
    # print(df_heatmap.columns)
    if val_base in df_heatmap.columns and val_source in df_heatmap.columns:
        df_heatmap["proportion"] = df_heatmap[val_source] / (df_heatmap[val_source] + df_heatmap[val_base])
    else:
        print(df_heatmap.head())
        raise KeyError(f"Values {val_base} and {val_source} not in DataFrame.")
        df_heatmap["proportion"] = 0.5  # Default to neutral if data is missing
    
    proportion = 0
    layer_i = 0
    earliest_layer = None
    
    # earliest layer is the first layer where the proportion of source exceeds 0.5
    while layer_i in df_heatmap.index:
        proportion = df_heatmap.iloc[layer_i]["proportion"]
        
        if earliest_layer is None:
            if proportion >= 0.5:
                earliest_layer = layer_i
        else:
            if proportion < 0.5:
                earliest_layer = None
        
        layer_i += 1
    return earliest_layer, df_heatmap.iloc[earliest_layer]["proportion"] if earliest_layer is not None else (None, None)

In [ ]:
def get_heatmap(model_short, column, base_vals, source_vals, postfix=""):
    heatmap = torch.zeros(len(src_src), len(tgt_tgt))
    for i, src_setting in enumerate(src_src):
        for j, tgt_setting in enumerate(tgt_tgt):
            probs_path = f"output/intervention/probs/block_noun-adj_{model_short}_{src_setting['src_lang_base']}-{tgt_setting['tgt_lang_base']}_{src_setting['src_lang_source']}-{tgt_setting['tgt_lang_source']}{postfix}.csv"
            try:
                probs_df = pd.read_csv(probs_path)
            except IOError:
                continue
            layer_val, prob = get_earliest_layer(probs_df, column, base_vals[i][j], source_vals[i][j])
            heatmap[i, j] = layer_val
    return heatmap

In [4]:
def plot_heatmap(hm, model_name="LLM", column_name="parameter"):
    # Convert heatmap tensor to numpy for Plotly
    heatmap_np = hm.numpy()

    # Prepare text annotations for each cell
    text = [[str(int(val)) for val in row] for row in heatmap_np]

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_np,
        x=tgt_labels,
        y=src_labels,
        text=text,
        texttemplate="%{text}",
        colorbar=dict(title="Earliest Layer >50%"),
        colorscale="Viridis"
    ))
    fig.update_layout(
        title=f"Earliest Layer with >50% Source {column_name} ({model_name})",
        xaxis_title="Target Setting",
        yaxis_title="Source Setting",
        autosize=False,
        width=700,
        height=600
    )
    fig.show()

In [ ]:
model_names = ["mgpt", "aya-expanse-8b", "llama-3-8b"]

# Language
lang_base_vals = []
lang_source_vals = []
for i, src_setting in enumerate(src_src):
    lang_base_vals.append([])
    lang_source_vals.append([])
    for j, tgt_setting in enumerate(tgt_tgt):
        lang_base_vals[i].append(tgt_setting['tgt_lang_base'])
        lang_source_vals[i].append(tgt_setting['tgt_lang_source'])

# Part of Speech
pos_base_vals = []
pos_source_vals = []
for i, src_setting in enumerate(src_src):
    pos_base_vals.append([])
    pos_source_vals.append([])
    for j, tgt_setting in enumerate(tgt_tgt):
        pos_base_vals[i].append(tgt2s[tgt_setting['tgt_lang_base']])
        pos_source_vals[i].append(tgt2s[tgt_setting['tgt_lang_source']])

print(pos_base_vals)
for model_name in model_names:
    heatmap_pos = get_heatmap(model_name, "part_of_speech", 14, base_vals=pos_base_vals, source_vals=pos_source_vals, postfix=postfix)
    heatmap_lang = get_heatmap(model_name, "language", 14, base_vals=lang_base_vals, source_vals=lang_source_vals, postfix=postfix)
    plot_heatmap(heatmap_pos, model_name=model_name, column_name="Part of Speech")
    plot_heatmap(heatmap_lang, model_name=model_name, column_name="Language")
    plot_heatmap(heatmap_lang-heatmap_pos, model_name=model_name, column_name="Difference Language-PoS")

[['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'adj', 'noun'], ['noun', 'adj', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'adj', 'noun', 'ad